AI-Based Movie Recommendation System

This project develops a content-based movie recommendation system that recommends similar movies based on their characteristics. The system will later be extended with a simple user-friendly interface.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Movies%20Recommendation.csv"

movies = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Number of movies:", len(movies))

movies.head()

Dataset loaded successfully!
Number of movies: 4760


,Movie_ID,Movie_Title,Movie_Genre,Movie_Language,Movie_Budget,Movie_Popularity,Movie_Release_Date,Movie_Revenue,Movie_Runtime,Movie_Vote,...,Movie_Homepage,Movie_Keywords,Movie_Overview,Movie_Production_House,Movie_Production_Country,Movie_Spoken_Language,Movie_Tagline,Movie_Cast,Movie_Crew,Movie_Director
0,1,Four Rooms,Crime Comedy,en,4000000,22.876230,09-12-1995,4300000,98.0,6.5,...,NaN,hotel new year's eve witch bet hotel room,It's Ted the Bellhop's first night on the job....,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]",Twelve outrageous guests. Four scandalous requ...,Tim Roth Antonio Banderas Jennifer Beals Madon...,"[{'name': 'Allison Anders', 'gender': 1, 'depa...",Allison Anders
1,2,Star Wars,Adventure Action Science Fiction,en,11000000,126.393695,25-05-1977,775398007,121.0,8.1,...,http://www.starwars.com/films/star-wars-episod...,android galaxy hermit death star lightsaber,Princess Leia is captured and held hostage by ...,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","A long time ago in a galaxy far, far away...",Mark Hamill Harrison Ford Carrie Fisher Peter ...,"[{'name': 'George Lucas', 'gender': 2, 'depart...",George Lucas
2,3,Finding Nemo,Animation Family,en,94000000,85.688789,30-05-2003,940335536,100.0,7.6,...,http://movies.disney.com/finding-nemo,father son relationship harbor underwater fish...,"Nemo, an adventurous young clownfish, is unexp...","[{""name"": ""Pixar Animation Studios"", ""id"": 3}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","There are 3.7 trillion fish in the ocean, they...",Albert Brooks Ellen DeGeneres Alexander Gould ...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton
3,4,Forrest Gump,Comedy Drama Romance,en,55000000,138.133331,06-07-1994,677945399,142.0,8.2,...,NaN,vietnam veteran hippie mentally disabled runni...,A man with a low IQ has accomplished great thi...,"[{""name"": ""Paramount Pictures"", ""id"": 4}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","The world will never be the same, once you've ...",Tom Hanks Robin Wright Gary Sinise Mykelti Wil...,"[{'name': 'Alan Silvestri', 'gender': 2, 'depa...",Robert Zemeckis
4,5,American Beauty,Drama,en,15000000,80.878605,15-09-1999,356296601,122.0,7.9,...,http://www.dreamworks.com/ab/,male nudity female nudity adultery midlife cri...,"Lester Burnham, a depressed suburban father in...","[{""name"": ""DreamWorks SKG"", ""id"": 27}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]",Look closer.,Kevin Spacey Annette Bening Thora Birch Wes Be...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes


In [5]:
print("Dataset columns:")
print(movies.columns.tolist())

print("\nDataset shape:")
print(movies.shape)

Dataset columns:
['Movie_ID', 'Movie_Title', 'Movie_Genre', 'Movie_Language', 'Movie_Budget', 'Movie_Popularity', 'Movie_Release_Date', 'Movie_Revenue', 'Movie_Runtime', 'Movie_Vote', 'Movie_Vote_Count', 'Movie_Homepage', 'Movie_Keywords', 'Movie_Overview', 'Movie_Production_House', 'Movie_Production_Country', 'Movie_Spoken_Language', 'Movie_Tagline', 'Movie_Cast', 'Movie_Crew', 'Movie_Director']

Dataset shape:
(4760, 21)


In [6]:
movies_features = movies[
    ['Movie_Genre', 'Movie_Keywords', 'Movie_Tagline', 'Movie_Cast', 'Movie_Director']
].fillna('')

movies['Combined_Features'] = (
    movies_features['Movie_Genre'] + ' ' +
    movies_features['Movie_Keywords'] + ' ' +
    movies_features['Movie_Tagline'] + ' ' +
    movies_features['Movie_Cast'] + ' ' +
    movies_features['Movie_Director']
)

print("Movie features combined successfully!")

movies[['Movie_Title', 'Combined_Features']].head()

Movie features combined successfully!


,Movie_Title,Combined_Features
0,Four Rooms,Crime Comedy hotel new year's eve witch bet ho...
1,Star Wars,Adventure Action Science Fiction android galax...
2,Finding Nemo,Animation Family father son relationship harbo...
3,Forrest Gump,Comedy Drama Romance vietnam veteran hippie me...
4,American Beauty,Drama male nudity female nudity adultery midli...


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

feature_matrix = tfidf.fit_transform(movies['Combined_Features'])

print("TF-IDF feature matrix created successfully!")
print("Matrix shape:", feature_matrix.shape)

TF-IDF feature matrix created successfully!
Matrix shape: (4760, 17020)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

print("Cosine similarity matrix created successfully!")
print("Similarity matrix shape:", similarity_matrix.shape)

Cosine similarity matrix created successfully!
Similarity matrix shape: (4760, 4760)


In [9]:
def recommend_movies(movie_title, number_of_recommendations=5):
    movie_title = movie_title.lower()

    matching_movies = movies[
        movies['Movie_Title'].str.lower() == movie_title
    ]

    if matching_movies.empty:
        return ["Movie not found. Please check the title and try again."]

    movie_index = matching_movies.index[0]

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommended_movies = []

    for index, score in similarity_scores[1:number_of_recommendations + 1]:
        recommended_movies.append(movies.iloc[index]['Movie_Title'])

    return recommended_movies

In [10]:
recommend_movies("Avatar")

['Alien',
 'Guardians of the Galaxy',
 'Aliens',
 'Star Trek Beyond',
 'Star Trek Into Darkness']

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output

movie_input = widgets.Text(
    placeholder='Enter a movie title',
    description='Movie:',
    style={'description_width': 'initial'}
)

recommend_button = widgets.Button(
    description='Get Recommendations'
)

output = widgets.Output()

def show_recommendations(button):
    with output:
        clear_output()

        movie_title = movie_input.value.strip()
        recommendations = recommend_movies(movie_title)

        print(f"Recommendations for: {movie_title}\n")

        for i, movie in enumerate(recommendations, start=1):
            print(f"{i}. {movie}")

recommend_button.on_click(show_recommendations)

display(movie_input, recommend_button, output)

Text(value='', description='Movie:', placeholder='Enter a movie title', style=DescriptionStyle(description_wid…

Button(description='Get Recommendations', style=ButtonStyle())

Output()

In [12]:
import difflib

def recommend_movies(movie_title, number_of_recommendations=5):
    movie_titles = movies['Movie_Title'].dropna().tolist()

    matches = difflib.get_close_matches(
        movie_title,
        movie_titles,
        n=1,
        cutoff=0.4
    )

    if not matches:
        return None, []

    matched_title = matches[0]
    movie_index = movies[movies['Movie_Title'] == matched_title].index[0]

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommended_movies = []

    for index, score in similarity_scores[1:number_of_recommendations + 1]:
        recommended_movies.append(movies.iloc[index]['Movie_Title'])

    return matched_title, recommended_movies

print("Recommendation function updated successfully!")

Recommendation function updated successfully!


In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

movie_input = widgets.Text(
    placeholder='Enter a movie title',
    description='Movie:',
    style={'description_width': 'initial'}
)

recommend_button = widgets.Button(
    description='Get Recommendations'
)

output = widgets.Output()

def show_recommendations(button):
    with output:
        clear_output()

        movie_title = movie_input.value.strip()
        matched_title, recommendations = recommend_movies(movie_title)

        if not recommendations:
            print("Movie not found. Please try another title.")
            return

        print(f"Closest match: {matched_title}")
        print("\nRecommended Movies:\n")

        for i, movie in enumerate(recommendations, start=1):
            print(f"{i}. {movie}")

recommend_button.on_click(show_recommendations)

display(movie_input, recommend_button, output)

Text(value='', description='Movie:', placeholder='Enter a movie title', style=DescriptionStyle(description_wid…

Button(description='Get Recommendations', style=ButtonStyle())

Output()

In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output

movie_input = widgets.Text(
    placeholder='Enter a movie title',
    description='Movie:',
    style={'description_width': 'initial'}
)

recommend_button = widgets.Button(
    description='Get Recommendations'
)

output = widgets.Output()

def show_recommendations(button):
    with output:
        clear_output()

        movie_title = movie_input.value.strip()
        matched_title, recommendations = recommend_movies(movie_title)

        if not recommendations:
            print("Movie not found. Please try another title.")
            return

        print(f"Closest match: {matched_title}")
        print("\nRecommended Movies:\n")

        for i, movie in enumerate(recommendations, start=1):
            print(f"{i}. {movie}")

recommend_button.on_click(show_recommendations)

display(movie_input, recommend_button, output)

Text(value='', description='Movie:', placeholder='Enter a movie title', style=DescriptionStyle(description_wid…

Button(description='Get Recommendations', style=ButtonStyle())

Output()

In [15]:
sample_movies = [
    "Avatar",
    "Iron Man",
    "The Dark Knight",
    "Titanic",
    "The Avengers",
    "Jurassic World"
]

movie_dropdown = widgets.Dropdown(
    options=sample_movies,
    description='Choose a Movie:',
    style={'description_width': 'initial'}
)

recommend_button2 = widgets.Button(
    description='Recommend Movies'
)

output2 = widgets.Output()

def dropdown_recommendations(button):
    with output2:
        clear_output()

        selected_movie = movie_dropdown.value
        matched_title, recommendations = recommend_movies(selected_movie)

        print(f"Because you selected: {matched_title}")
        print("\nYou may also like:\n")

        for i, movie in enumerate(recommendations, start=1):
            print(f"{i}. {movie}")

recommend_button2.on_click(dropdown_recommendations)

display(movie_dropdown, recommend_button2, output2)

Dropdown(description='Choose a Movie:', options=('Avatar', 'Iron Man', 'The Dark Knight', 'Titanic', 'The Aven…

Button(description='Recommend Movies', style=ButtonStyle())

Output()